In [0]:
%pip install xgboost==3.0.5

# Reinicia o Python para o pacote ficar visível

dbutils.library.restartPython()

In [0]:
import mlflow

MODEL_NAME = "workspace.default.hackathon_databricks_model"

# Listar versões disponíveis
client = mlflow.MlflowClient(registry_uri="databricks-uc")
versions = client.search_model_versions(filter_string=f"name='{MODEL_NAME}'")
for v in sorted(versions, key=lambda x: int(x.version)):
    print(f"v{v.version}: status={v.status}")

# Carregar a versão mais recente (v3)
model = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}/3")
print("\nModelo carregado com sucesso:", MODEL_NAME, "v3")

# Esquema de entrada esperado pelo modelo
print("\nInput schema:")
print(model.metadata.get_input_schema())

In [0]:
# Carregar a tabela workspace.default.train_classificado em um DataFrame Pandas
df = spark.table("workspace.default.train_classificado").toPandas()

print(f"Total de linhas: {len(df)}")
print(f"Total de colunas: {df.shape[1]}")

# Amostra dos dados
df.head()


In [0]:
# ============================================================
# FILTRAGEM DE OBSERVACOES
# ============================================================

def filtrar_observacoes_invalidas(df):
    """
    Exclui observacoes onde simultaneamente:

        tenure == 'unknown'
        AND
        TotalCharges == NULL

    Essa regra nao depende de estatisticas da base e, portanto, pode ser executada antes do split.
    Retorna um novo DataFrame sem modificar o original.
    """

    df = df.copy()

    tenure_unknown = (df["tenure"].astype("string").str.strip().str.lower().eq("unknown"))

    total_charges_null = (df["TotalCharges"].isna())

    mascara_excluir = (tenure_unknown & total_charges_null)

    return (df.loc[~mascara_excluir].copy().reset_index(drop=True))


In [0]:
df = filtrar_observacoes_invalidas(df)
df

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)

# ---------------------------------------------------------------
# 1) Preparar os dados de entrada conforme o esquema do modelo
# ---------------------------------------------------------------
schema = model.metadata.get_input_schema()
feature_cols = [c.name for c in schema.inputs]
X = df[feature_cols].copy()

# Coluna com o rótulo verdadeiro (ajuste se necessário)
label_col = next(c for c in df.columns if "churn" in c.lower())
y_true = (
    df[label_col].astype(str).str.strip().str.lower()
    .map({"yes": 1, "true": 1, "1": 1, "no": 0, "false": 0, "0": 0})
)
assert y_true.notna().all(), f"Valores inesperados em {label_col}: {df[label_col].unique()}"

# ---------------------------------------------------------------
# 2) Obter a probabilidade de Churn para cada linha
# ---------------------------------------------------------------
raw_pred = model.predict(X)
if isinstance(raw_pred, pd.DataFrame):
    raw_pred = raw_pred.iloc[:, 0]

p_churn = np.asarray(raw_pred, dtype=float)
# Se o modelo devolver rótulos (0/1) em vez de probabilidade, normaliza:
if p_churn.min() >= 0 and p_churn.max() <= 1 and set(np.unique(p_churn)) <= {0.0, 1.0}:
    p_churn = p_churn  # já está no espaço [0,1]; assumimos probabilidade

print(f"AUC-ROC do modelo: {roc_auc_score(y_true, p_churn):.4f}\n")

# ---------------------------------------------------------------
# 3) Varredura de cortes candidatos
# ---------------------------------------------------------------
thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
rows = []
for t in thresholds:
    y_pred = (p_churn >= t).astype(int)
    rows.append({
        "threshold": t,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "youden_j": None,  # calculado abaixo via ROC
    })

# Youden's J (recall + especificidade - 1) direto da curva ROC
fpr, tpr, roc_thr = roc_curve(y_true, p_churn)
j_scores = tpr - fpr
best_j_idx = j_scores.argmax()

# Melhor corte por F1
best_f1_idx = np.argmax([r["f1"] for r in rows])
best_f1 = rows[best_f1_idx]

# Melhor corte por Youden's J
best_youden = {
    "threshold": float(roc_thr[best_j_idx]),
    "youden_j": float(j_scores[best_j_idx]),
    "tpr": float(tpr[best_j_idx]),
    "fpr": float(fpr[best_j_idx]),
}

# Melhor corte pela curva Precision-Recall
prec_arr, rec_arr, pr_thr = precision_recall_curve(y_true, p_churn)
f1_arr = 2 * prec_arr * rec_arr / np.where(prec_arr + rec_arr == 0, 1e-12, prec_arr + rec_arr)
best_pr_idx = f1_arr.argmax()
best_pr_thr = float(pr_thr[min(best_pr_idx, len(pr_thr) - 1)])

print("=== Melhores cortes ===")
print(f"Por F1-score:        corte = {best_f1['threshold']:.2f}  (F1={best_f1['f1']:.4f}, precisão={best_f1['precision']:.4f}, recall={best_f1['recall']:.4f})")
print(f"Por Youden's J:     corte = {best_youden['threshold']:.4f}  (J={best_youden['youden_j']:.4f})")
print(f"Por curva P-R:      corte = {best_pr_thr:.4f}  (F1={f1_arr[best_pr_idx]:.4f})")

# ---------------------------------------------------------------
# 4) Curva de F1 vs. corte (escolha interativa do negócio)
# ---------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))

axes[0].plot([r["threshold"] for r in rows], [r["f1"] for r in rows], label="F1")
axes[0].axvline(best_f1["threshold"], color="red", ls="--", label=f"Melhor F1 (corte={best_f1['threshold']:.2f})")
axes[0].set_xlabel("Corte de probabilidade"); axes[0].set_ylabel("F1-score"); axes[0].legend(); axes[0].set_title("F1 vs. corte")

axes[1].plot([r["threshold"] for r in rows], [r["precision"] for r in rows], label="Precisão")
axes[1].plot([r["threshold"] for r in rows], [r["recall"] for r in rows], label="Recall")
axes[1].axvline(best_f1["threshold"], color="red", ls="--")
axes[1].set_xlabel("Corte de probabilidade"); axes[1].legend(); axes[1].set_title("Precisão/Recall vs. corte")

axes[2].plot(fpr, tpr, label=f"ROC (AUC={roc_auc_score(y_true, p_churn):.3f})")
axes[2].scatter(best_youden["fpr"], best_youden["tpr"], color="red", zorder=5, label=f"Youden (corte={best_youden['threshold']:.3f})")
axes[2].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[2].set_xlabel("FPR"); axes[2].set_ylabel("TPR"); axes[2].legend(); axes[2].set_title("Curva ROC")

plt.tight_layout()
plt.show()

# ---------------------------------------------------------------
# 5) Tabela completa e relatório no corte escolhido
# ---------------------------------------------------------------
eval_df = pd.DataFrame(rows).drop(columns=["youden_j"])
display(eval_df.sort_values("f1", ascending=False).head(10))

CHOSEN_THRESHOLD = best_f1["threshold"]  
y_final = (p_churn >= CHOSEN_THRESHOLD).astype(int)
print(f"\n=== Desempenho no corte escolhido: {CHOSEN_THRESHOLD:.2f} ===")
print(confusion_matrix(y_true, y_final))
print(classification_report(y_true, y_final, target_names=["No", "Yes"]))

In [0]:
# ============================================================
# AVALIACAO DE CORTES: 0.1 a 0.9, passo 0.1
# Reutiliza y_true e p_churn calculados na celula anterior
# ============================================================

# Taxa de resposta da base (proporcao de churn) usada como corte adicional

taxa_resposta = float(y_true.mean())
print(f"Taxa de resposta da base (churn rate): {taxa_resposta:.4f} ({taxa_resposta*100:.2f}%)")

thresholds = np.sort(np.append(np.round(np.arange(0.1, 0.95, 0.1), 2), round(taxa_resposta, 4)))

rows = []
for t in thresholds:
    y_pred = (p_churn >= t).astype(int)
    rows.append({
        "threshold": t,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    })

thresholds_df = pd.DataFrame(rows)

print("=== Metricas por corte (0.1 a 0.9, incluindo a taxa de resposta da base) ===")
display(thresholds_df)

linha_base = thresholds_df.loc[thresholds_df["threshold"] == round(taxa_resposta, 4)]
if not linha_base.empty:
    lb = linha_base.iloc[0]
    print(f"\nNo corte = taxa de resposta ({lb['threshold']:.4f}): acuracia={lb['accuracy']:.4f}, precisao={lb['precision']:.4f}, recall={lb['recall']:.4f}, F1={lb['f1']:.4f}")

print("\n=== Matrizes de confusao por corte ===")
for t in thresholds:
    y_pred = (p_churn >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    print(f"Corte {t:.1f}: TN={tn:5d} | FP={fp:4d} | FN={fn:4d} | TP={tp:4d}")

best = thresholds_df.loc[thresholds_df["f1"].idxmax()]
print(f"\nMelhor corte por F1: {best['threshold']:.1f} (F1={best['f1']:.4f}, precisao={best['precision']:.4f}, recall={best['recall']:.4f})")

# Visualizacao das metricas em funcao do corte
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(thresholds_df["threshold"], thresholds_df["f1"], marker="o", label="F1")
axes[0].plot(thresholds_df["threshold"], thresholds_df["accuracy"], marker="s", label="Acuracia")
axes[0].axvline(best["threshold"], color="red", ls="--", label=f"Melhor F1 (corte={best['threshold']:.1f})")
axes[0].axvline(taxa_resposta, color="gray", ls=":", label=f"Taxa de resposta ({taxa_resposta:.2f})")
axes[0].set_xlabel("Corte de probabilidade")
axes[0].set_ylabel("Metrica")
axes[0].set_xticks(thresholds)
axes[0].legend()
axes[0].set_title("F1 e Acuracia vs. corte")

axes[1].plot(thresholds_df["threshold"], thresholds_df["precision"], marker="o", label="Precisao")
axes[1].plot(thresholds_df["threshold"], thresholds_df["recall"], marker="s", label="Recall")
axes[1].set_xlabel("Corte de probabilidade")
axes[1].set_ylabel("Metrica")
axes[1].set_xticks(thresholds)
axes[1].legend()
axes[1].set_title("Precisao e Recall vs. corte")

plt.tight_layout()
plt.show()

In [0]:
# Metricas e Matriz de Confusao para um corte de probabilidade especificado
# Reutiliza y_true e p_churn calculados nas celulas anteriores

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report,
)
import matplotlib.pyplot as plt

# Para considerar, de forma proporcional, as metricas Precision e Recall, optamos pelo corte 0.6.
# No entanto, a depender da avaliacao do Negocio, essa linha de corte pode ser alterada. Exemplo:
# - Se evitar falsos positivos (churn incorretamente previsto) for mais importante, aumentar o corte.
# - Se evitar falsos negativos (deixar de prever um churn) for mais importante, diminuir o corte.

threshold = 0.6

y_pred = (p_churn >= threshold).astype(int)

matriz = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(matriz, cmap="Blues")

ax.set_xticks([0, 1], ["Previsto No", "Previsto Yes"])
ax.set_yticks([0, 1], ["Real No", "Real Yes"])
ax.set_title(f"Matriz de Confusao (threshold = {threshold:.2f})")

for i in range(matriz.shape[0]):
    for j in range(matriz.shape[1]):
        ax.text(j, i, f"{matriz[i, j]:d}", ha="center", va="center",
                color="white" if matriz[i, j] > matriz.max() / 2 else "black",
                fontsize=12)

fig.colorbar(im, ax=ax, label="Observacoes")
plt.tight_layout()
plt.show()

print(f"=== Metricas para threshold = {threshold:.2f} ===")
print(f"Acuracia:  {accuracy_score(y_true, y_pred):.4f}")
print(f"Precisao:  {precision_score(y_true, y_pred, zero_division=0):.4f}")
print(f"Recall:    {recall_score(y_true, y_pred, zero_division=0):.4f}")
print(f"F1-score:  {f1_score(y_true, y_pred, zero_division=0):.4f}")

print(f"\n=== Matriz de Confusao (threshold = {threshold:.2f}) ===")
print(f"              Previsto No | Previsto Yes")
print(f"Real No      {tn:10d} | {fp:10d}")
print(f"Real Yes     {fn:10d} | {tp:10d}")

print("\n=== Relatorio de classificacao ===")
print(classification_report(y_true, y_pred, target_names=["No", "Yes"]))



In [0]:
# ============================================================
# RANKING DE IMPORTANCIA DAS VARIAVEIS DO MODELO (SHAP)
# ============================================================

%pip install shap

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

# Esquema de entrada do modelo (mesma fonte usada na avaliacao)
schema = model.metadata.get_input_schema()
feature_cols = [c.name for c in schema.inputs]
X = df[feature_cols].copy()

# ---------------------------------------------------------------
# 1) SHAP nativo para XGBoost (TreeExplainer: rapido e exato)
#    O valor SHAP de uma variavel em uma observacao e sua
#    contribuicao marginal para o desvio da previsao em relacao
#    ao valor base (E[output]).
# ---------------------------------------------------------------
try:
    xgb_model = mlflow.xgboost.load_model(f"models:/{MODEL_NAME}/3")
    booster = xgb_model.get_booster() if hasattr(xgb_model, "get_booster") else xgb_model

    explainer = shap.TreeExplainer(booster)
    shap_values = explainer.shap_values(X)

    # Normaliza formatos de saida (multiclasse retorna lista; binario pode retornar 3D)
    if isinstance(shap_values, list):
        shap_values = shap_values[-1]  # classe positiva
    shap_values = np.asarray(shap_values)
    if shap_values.ndim == 3:
        shap_values = shap_values[..., -1]  # classe positiva

    # -----------------------------------------------------------
    # Ranking: media do |SHAP| por variavel
    # -----------------------------------------------------------
    imp_df = pd.DataFrame({
        "feature": feature_cols,
        "shap_importancia_media": np.abs(shap_values).mean(axis=0),
    }).sort_values("shap_importancia_media", ascending=False).reset_index(drop=True)
    imp_df.insert(0, "rank", imp_df.index + 1)
    total = imp_df["shap_importancia_media"].sum()
    imp_df["shap_pct"] = 100 * imp_df["shap_importancia_media"] / total if total > 0 else 0.0

    print("=== Ranking de importancia das variaveis (SHAP - media do |valor SHAP|) ===")
    display(imp_df[["rank", "feature", "shap_importancia_media", "shap_pct"]])

    # -----------------------------------------------------------
    # Graficos: barras (importancia media) e beeswarm (efeito por observacao)
    # -----------------------------------------------------------
    top_n = min(20, len(imp_df))

    fig, ax = plt.subplots(figsize=(10, max(5, 0.4 * top_n)))
    top = imp_df.head(top_n).iloc[::-1]
    ax.barh(top["feature"], top["shap_importancia_media"], color="steelblue")
    ax.set_xlabel("Media do |valor SHAP|")
    ax.set_title(f"Top {top_n} variaveis por importancia SHAP")
    plt.tight_layout()
    plt.show()

    print("\n=== Beeswarm SHAP (valor da variavel x impacto na previsao de churn) ===")
    shap.summary_plot(
        shap_values, X,
        max_display=top_n,
        plot_size=(10, max(4, 0.35 * top_n)),
        show=False,
    )
    plt.show()

except Exception as e:
    # ---------------------------------------------------------------
    # Fallback: SHAP sobre o modelo pyfunc (independe do flavor)
    # Usa um fundo (background) amostral para o masker e explica
    # uma amostra das observacoes para viabilizar o custo computacional
    # ---------------------------------------------------------------
    print(f"Nao foi possivel usar o TreeExplainer ({e}).")
    print("Usando shap.Explainer sobre o modelo pyfunc.\n")

    # O masker tabular do SHAP (np.isclose) nao suporta colunas de texto.
    # Ponte: codifica as colunas string como inteiros no espaco do masker e
    # decodifica (com os dtypes originais) dentro do predict.
    colunas_str = X.select_dtypes(include=['object']).columns.tolist()
    categorias = {c: X[c].astype('category').cat.categories for c in colunas_str}
    mediana_tc = X["TotalCharges"].median()

    def codificar(X_input):
        Xc = X_input.copy()
        for c in colunas_str:
            Xc[c] = pd.Categorical(X_input[c], categories=categorias[c]).codes
        return Xc

    def decodificar(X_cod):
        if not isinstance(X_cod, pd.DataFrame):
            X_cod = pd.DataFrame(np.atleast_2d(np.asarray(X_cod)), columns=feature_cols)
        Xd = X_cod.copy()
        for c in colunas_str:
            Xd[c] = np.asarray(categorias[c])[np.asarray(Xd[c]).astype(int)]
        for c in feature_cols:
            if c not in colunas_str:
                Xd[c] = Xd[c].astype(X[c].dtype)
        return Xd

    def predict_proba(X_cod):
        X_input = decodificar(X_cod)
        # A transformacao interna do modelo quebra quando tenure='unknown' e
        # TotalCharges e nulo — combinacao que as mascaras do SHAP podem criar
        m = (X_input["tenure"].astype(str).str.strip().str.lower().eq("unknown")
             & X_input["TotalCharges"].isna())
        if m.any():
            X_input.loc[m, "TotalCharges"] = mediana_tc
        out = model.predict(X_input[feature_cols])
        if isinstance(out, pd.DataFrame):
            out = out.iloc[:, 0]
        return np.asarray(out, dtype=float)

    n_bg = min(100, len(X))
    n_explain = min(200, len(X))

    X_bg = codificar(X.sample(n=n_bg, random_state=42))
    X_explain = codificar(X.sample(n=n_explain, random_state=123))

    masker = shap.maskers.Independent(X_bg, max_samples=n_bg)
    explainer = shap.Explainer(predict_proba, masker, algorithm="permutation")
    shap_exp = explainer(X_explain, max_evals=2 * len(feature_cols) + 1)

    shap_values = np.asarray(shap_exp.values)

    imp_df = pd.DataFrame({
        "feature": feature_cols,
        "shap_importancia_media": np.abs(shap_values).mean(axis=0),
    }).sort_values("shap_importancia_media", ascending=False).reset_index(drop=True)
    imp_df.insert(0, "rank", imp_df.index + 1)

    print("=== Ranking de importancia por SHAP (amostra explicada) ===")
    display(imp_df)

    top_n = min(20, len(imp_df))

    print(f"\n=== Beeswarm SHAP (explicando {n_explain} observacoes) ===")
    shap.plots.beeswarm(shap_exp, max_display=top_n)
    plt.show()
